# 03 — Evolutionary scores (ESM-2 650M)

dualde infers a single sequence model: **ESM-2 650M** (`esm2_t33_650M_UR50D`).
Inference runs **only if a CUDA GPU is available**; otherwise this notebook
loads the precomputed table.

**EVmutation** scores and the **coupling graph** are not computed here. Obtain
both from the [EVcouplings webserver](https://evcouplings.org). For NUDT15 they
are already included in `scores/evolutionary.csv` and
`couplings/b0.5_CouplingScoresCompared_all.csv`.


## Resolve scores (GPU scan or cached table)


In [ ]:
from IPython.display import display
import pandas as pd

from dualde.device import cuda_is_available
from dualde.mapping import read_fasta
from dualde.paths import example_dir
from dualde.scores_esm import ESM2_650M_COLUMN, run_esm2_650m

EXAMPLE = example_dir("NUDT15_HUMAN")
OUT = EXAMPLE / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
CACHE = EXAMPLE / "scores" / "evolutionary.csv"

cached = pd.read_csv(CACHE)

if cuda_is_available():
    print("CUDA GPU detected; running ESM-2 650M.")
    _, sequence = read_fasta(EXAMPLE / "NUDT15_HUMAN.fasta")
    try:
        evo = run_esm2_650m(sequence, protein_id="NUDT15_HUMAN")
        if "EVmutation" in cached.columns:
            evo = evo.merge(
                cached[["mutation", "EVmutation"]].drop_duplicates(),
                on="mutation",
                how="left",
            )
    except ImportError as err:
        print(err)
        print("Falling back to the precomputed table.")
        evo = cached
else:
    print("No CUDA GPU; loading precomputed ESM-2 650M scores.")
    evo = cached

evo.to_csv(OUT / "evolutionary.csv", index=False)
print("wrote", OUT / "evolutionary.csv")


## Inspect the score table


In [ ]:
print("shape:", evo.shape)
print("columns:", list(evo.columns))
evo.head()


In [ ]:
esm_cols = [c for c in evo.columns if c.startswith("ESM2")]
print("ESM-2 columns (dualde uses ESM2_650M):", esm_cols)
print("missingness:")
cols = [c for c in [ESM2_650M_COLUMN, "EVmutation"] if c in evo.columns]
display(evo[cols].isna().mean().to_frame("frac_na"))
if ESM2_650M_COLUMN in evo.columns:
    display(evo[[ESM2_650M_COLUMN]].describe().T)
else:
    print("WARNING: ESM2_650M is absent from this table.")


In [ ]:
site_cols = ["pos", "ref", "aa_to", "mutation"] + [c for c in [ESM2_650M_COLUMN, "EVmutation"] if c in evo.columns]
site = evo.loc[evo["pos"] == 45, site_cols]
print("position 45 (n =", len(site), ")")
site.head(10)


## EVcouplings webserver (new proteins)

1. Submit the FASTA at [evcouplings.org](https://evcouplings.org).
2. Merge the single-mutant / EVmutation table onto `mutation` (for example `Q144A`) as column `EVmutation`.
3. Place `*CouplingScoresCompared_all.csv` (`i`, `j`, `probability`, `dist`) where notebook 06 can read it.

Optional GPU install: `pip install -e ".[evo]"` and `export HF_HOME=$PWD/.cache/huggingface`.
